<a href="https://colab.research.google.com/github/bvsvntv/ai-ml-learning/blob/main/notebooks/lang-graph/subgraph_with_shared_states.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -U langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 587.1/587.1 kB 9.3 MB/s  0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.7.4
    Uninstalling langchain-core-1.7.4:
      Successfully uninstalled langchain-core-1.7.4
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [langchain-openai]


In [2]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_openai import ChatOpenAI
from google.colab import userdata

In [4]:
api_key = userdata.get("OPENAI_API_KEY")

In [5]:
class ParentState(TypedDict):
  question: str
  answer_eng: str
  answer_nep: str

In [6]:
llm = ChatOpenAI(api_key=api_key, model="gpt-6-luna")

### SubGraph

In [9]:
def translate_text(state: ParentState):
  prompt = f"""
  Translate the following text to Nepali.
  Keep it natural and clear. Do not add extra content.

  Text:
  {state["answer_eng"]}
  """.strip()

  translated_text = llm.invoke(prompt).content
  return {"answer_nep": translated_text}

In [10]:
subgraph = StateGraph(ParentState)

subgraph.add_node("translate_text", translate_text)

subgraph.add_edge(START, "translate_text")
subgraph.add_edge("translate_text", END)

sub_workflow = subgraph.compile()

### ParentGraph

In [11]:
def generate_answer(state: ParentState):
  prompt = f"You are a helpful assistant. Answer the following question: {state["question"]}"

  answer = llm.invoke(prompt).content
  return {"answer_eng": answer}

In [12]:
graph = StateGraph(ParentState)

graph.add_node("answer", generate_answer)
graph.add_node("translate", sub_workflow)

graph.add_edge(START, "answer")
graph.add_edge("answer", "translate")
graph.add_edge("translate", END)

parent_workflow = graph.compile()

In [13]:
initial_state = {
    "question": "What is photosynthesis?"
}
result = parent_workflow.invoke(initial_state)

print(result)

{'question': 'What is photosynthesis?', 'answer_eng': 'Photosynthesis is the process by which plants, algae, and some bacteria use light energy to turn carbon dioxide and water into sugars (such as glucose), releasing oxygen as a byproduct.\n\nA simplified equation is: **carbon dioxide + water + light → glucose + oxygen**.', 'answer_nep': 'प्रकाश संश्लेषण भनेको बिरुवा, शैवाल र केही ब्याक्टेरियाले प्रकाश ऊर्जाको प्रयोग गरेर कार्बन डाइअक्साइड र पानीलाई चिनी (जस्तै ग्लुकोज) मा रूपान्तरण गर्ने प्रक्रिया हो। यस प्रक्रियामा उप-उत्पादनका रूपमा अक्सिजन निस्कन्छ।\n\nयसको सरल समीकरण यस्तो छ: **कार्बन डाइअक्साइड + पानी + प्रकाश → ग्लुकोज + अक्सिजन**।'}
